# DeepFilterNet-Light V2 — Enhanced Lightweight Speech Enhancement

**V2 Improvements over V1:**
- Increased model capacity (still < 1M params)
- Longer segment training (3s vs 2s)
- Optimized regularization
- Extended training epochs

**Key Features:**
- **< 1M parameters** for real-time smartphone deployment
- **Dual-path architecture**: ERB band processing + Deep Filtering
- **GRU-based temporal modeling** for capturing long-range dependencies
- **SDR loss only** for cleaner optimization
- **Learning rate scheduling** (warmup + cosine annealing) to avoid early plateau
- **Target: SNR > 15 dB**

**V2 Changes:**
- `hidden_dim`: 64 → 80
- `gru_dim`: 96 → 128
- `n_erb_bands`: 32 → 40
- `df_bins`: 96 → 128
- `df_order`: 3 → 5
- `SEGMENT_SECONDS`: 2.0 → 3.0
- `BATCH_SIZE`: 16 → 12
- `WEIGHT_DECAY`: 1e-4 → 5e-5
- `EPOCHS`: 150 → 200

In [1]:
import os
import re
import glob
import math
import json
import shutil
from dataclasses import dataclass
from typing import Optional, Tuple, List

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset

from tqdm.auto import tqdm

try:
    import torchaudio
    _HAS_TORCHAUDIO = True
except ModuleNotFoundError:
    torchaudio = None
    _HAS_TORCHAUDIO = False

try:
    import soundfile as sf
    _HAS_SOUNDFILE = True
except ModuleNotFoundError:
    sf = None
    _HAS_SOUNDFILE = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print('Device:', DEVICE)
print('Torch:', torch.__version__)

Device: cpu
Torch: 2.10.0


/Users/emonchowdhury/Desktop/Phase 2/av_zoom/audio/models/custom_model_1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## STFT Utilities

In [2]:
# STFT parameters optimized for 16kHz speech
N_FFT = 512
HOP_LENGTH = 128
WIN_LENGTH = 512


def get_window(win_length: int, device: torch.device) -> torch.Tensor:
    """Get sqrt-Hann window for STFT/iSTFT."""
    return torch.sqrt(torch.hann_window(win_length, periodic=True, device=device))


def stft(wav: torch.Tensor, device: torch.device = None) -> torch.Tensor:
    """Compute STFT. Input: [B, T] or [T]. Output: complex [B, F, T] or [F, T]."""
    if device is None:
        device = wav.device
    window = get_window(WIN_LENGTH, device)
    
    squeeze = wav.dim() == 1
    if squeeze:
        wav = wav.unsqueeze(0)
    
    spec = torch.stft(
        wav,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=window,
        center=True,
        return_complex=True,
    )
    
    if squeeze:
        spec = spec.squeeze(0)
    return spec


def istft(spec: torch.Tensor, device: torch.device = None, length: int = None) -> torch.Tensor:
    """Compute iSTFT. Input: complex [B, F, T] or [F, T]. Output: [B, T] or [T]."""
    if device is None:
        device = spec.device
    window = get_window(WIN_LENGTH, device)
    
    squeeze = spec.dim() == 2
    if squeeze:
        spec = spec.unsqueeze(0)
    
    wav = torch.istft(
        spec,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=window,
        center=True,
        length=length,
    )
    
    if squeeze:
        wav = wav.squeeze(0)
    return wav

## DeepFilterNet-Light V2 Model

Enhanced lightweight architecture:
- **ERB Encoder**: 40 ERB bands (vs 32 in V1)
- **Temporal GRU**: 128 hidden units (vs 96 in V1)
- **Deep Filter Module**: 5-frame filtering with 128 bins
- **< 1M parameters** target

In [3]:
class GroupedLinear(nn.Module):
    """Grouped linear layer for parameter efficiency."""
    def __init__(self, in_features: int, out_features: int, groups: int = 1, bias: bool = True):
        super().__init__()
        assert in_features % groups == 0 and out_features % groups == 0
        self.groups = groups
        self.in_per_group = in_features // groups
        self.out_per_group = out_features // groups
        
        self.weight = nn.Parameter(torch.randn(groups, self.out_per_group, self.in_per_group) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [..., in_features]
        batch_shape = x.shape[:-1]
        x = x.view(*batch_shape, self.groups, self.in_per_group)
        x = torch.einsum('...gi,goi->...go', x, self.weight)
        x = x.view(*batch_shape, -1)
        if self.bias is not None:
            x = x + self.bias
        return x


class ConvGLU(nn.Module):
    """Conv1D with Gated Linear Unit activation."""
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3, groups: int = 1):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch * 2, kernel_size, padding=kernel_size // 2, groups=groups)
        self.out_ch = out_ch
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x, gate = x.chunk(2, dim=1)
        return x * torch.sigmoid(gate)


class ERBEncoder(nn.Module):
    """Encode magnitude spectrum to ERB band features."""
    def __init__(
        self,
        n_freqs: int,
        n_erb_bands: int,
        hidden_dim: int,
        n_layers: int = 2,
    ):
        super().__init__()
        self.n_erb_bands = n_erb_bands
        
        # Learnable ERB projection (more flexible than fixed filterbank)
        self.erb_proj = nn.Linear(n_freqs, n_erb_bands)
        
        # Feature extraction
        layers = []
        in_dim = n_erb_bands
        for i in range(n_layers):
            out_dim = hidden_dim if i == n_layers - 1 else n_erb_bands * 2
            layers.append(ConvGLU(in_dim, out_dim, kernel_size=3))
            layers.append(nn.BatchNorm1d(out_dim))
            in_dim = out_dim
        self.layers = nn.Sequential(*layers)
    
    def forward(self, mag: torch.Tensor) -> torch.Tensor:
        # mag: [B, Freq, T]
        # Apply log compression
        mag = torch.log1p(mag * 10)  # Learnable compression
        
        # Project to ERB bands: [B, Freq, T] -> [B, T, Freq] -> [B, T, E] -> [B, E, T]
        erb = self.erb_proj(mag.transpose(1, 2)).transpose(1, 2)
        
        # Extract features
        return self.layers(erb)


class TemporalGRU(nn.Module):
    """Efficient GRU for temporal modeling."""
    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False,  # Causal for real-time
        )
        self.proj = nn.Linear(hidden_dim, input_dim) if hidden_dim != input_dim else nn.Identity()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, T] -> [B, T, C]
        x = x.transpose(1, 2)
        x, _ = self.gru(x)
        x = self.proj(x)
        return x.transpose(1, 2)


class DeepFilterModule(nn.Module):
    """Generate complex deep filters for multi-frame filtering."""
    def __init__(
        self,
        hidden_dim: int,
        n_freqs: int,
        df_order: int = 5,  # V2: increased from 3 to 5
        df_bins: int = 128,  # V2: increased from 96 to 128
    ):
        super().__init__()
        self.df_order = df_order
        self.df_bins = df_bins
        self.n_freqs = n_freqs
        
        # Deep filter coefficients: 2 (real/imag) * df_order * df_bins
        df_coef_dim = 2 * df_order * df_bins
        
        # Gain for all frequency bins
        self.gain_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_freqs),
            nn.Sigmoid(),
        )
        
        # Deep filter coefficients for low frequencies
        self.df_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, df_coef_dim),
        )
        
        # Initialize for identity-like behavior
        nn.init.zeros_(self.df_proj[-1].weight)
        nn.init.zeros_(self.df_proj[-1].bias)
    
    def forward(
        self,
        features: torch.Tensor,
        spec: torch.Tensor,
    ) -> torch.Tensor:
        """Apply deep filtering.
        
        Args:
            features: [B, C, T] hidden features
            spec: [B, Freq, T] complex input spectrum
        
        Returns:
            enhanced: [B, Freq, T] complex enhanced spectrum
        """
        B, n_freq, T = spec.shape
        
        # features: [B, C, T] -> [B, T, C]
        feat = features.transpose(1, 2)
        
        # Compute gains: [B, T, Freq]
        gains = self.gain_proj(feat)  # [B, T, Freq]
        gains = gains.transpose(1, 2)  # [B, Freq, T]
        
        # Compute DF coefficients: [B, T, 2 * df_order * df_bins]
        df_coefs = self.df_proj(feat)
        df_coefs = df_coefs.view(B, T, 2, self.df_order, self.df_bins)
        df_real = df_coefs[:, :, 0]  # [B, T, df_order, df_bins]
        df_imag = df_coefs[:, :, 1]
        
        # Apply gain to full spectrum
        enhanced = spec * gains
        
        # Apply deep filtering to low frequencies
        # Pad spectrum for causal filtering
        spec_padded = F.pad(spec[:, :self.df_bins, :], (self.df_order - 1, 0))
        
        # Multi-frame filtering
        df_out = torch.zeros(B, self.df_bins, T, dtype=spec.dtype, device=spec.device)
        
        for k in range(self.df_order):
            # Get frame at offset k
            frame = spec_padded[:, :, self.df_order - 1 - k:self.df_order - 1 - k + T]  # [B, df_bins, T]
            
            # Complex multiplication with DF coefficients
            coef_r = df_real[:, :, k, :].transpose(1, 2)  # [B, df_bins, T]
            coef_i = df_imag[:, :, k, :].transpose(1, 2)
            
            frame_r = frame.real
            frame_i = frame.imag
            
            # (a + bi)(c + di) = (ac - bd) + (ad + bc)i
            out_r = coef_r * frame_r - coef_i * frame_i
            out_i = coef_r * frame_i + coef_i * frame_r
            
            df_out = df_out + torch.complex(out_r, out_i)
        
        # Blend DF output with gain-only output for low frequencies
        # Use softer blending to allow DF to contribute
        alpha = 0.5  # Blend factor
        enhanced[:, :self.df_bins, :] = alpha * df_out + (1 - alpha) * enhanced[:, :self.df_bins, :]
        
        return enhanced


class DeepFilterNetLightV2(nn.Module):
    """Lightweight DeepFilterNet V2 for real-time speech enhancement.
    
    V2 improvements:
    - hidden_dim: 64 -> 80
    - gru_dim: 96 -> 128
    - n_erb_bands: 32 -> 40
    - df_bins: 96 -> 128
    - df_order: 3 -> 5
    
    Target: < 1M parameters
    """
    def __init__(
        self,
        n_fft: int = 512,
        n_erb_bands: int = 40,      # V2: increased from 32
        hidden_dim: int = 80,        # V2: increased from 64
        gru_dim: int = 128,          # V2: increased from 96
        gru_layers: int = 2,
        df_order: int = 5,           # V2: increased from 3
        df_bins: int = 128,          # V2: increased from 96
        dropout: float = 0.1,
    ):
        super().__init__()
        self.n_fft = n_fft
        self.n_freqs = n_fft // 2 + 1
        
        # ERB encoder
        self.erb_encoder = ERBEncoder(
            n_freqs=self.n_freqs,
            n_erb_bands=n_erb_bands,
            hidden_dim=hidden_dim,
            n_layers=2,
        )
        
        # Temporal modeling
        self.temporal_gru = TemporalGRU(
            input_dim=hidden_dim,
            hidden_dim=gru_dim,
            num_layers=gru_layers,
            dropout=dropout,
        )
        
        # Deep filter module
        self.deep_filter = DeepFilterModule(
            hidden_dim=hidden_dim,
            n_freqs=self.n_freqs,
            df_order=df_order,
            df_bins=min(df_bins, self.n_freqs),
        )
        
        # Parameter count check
        n_params = sum(p.numel() for p in self.parameters())
        print(f"DeepFilterNetLightV2: {n_params / 1e6:.3f}M parameters")
        if n_params > 1_000_000:
            print(f"  WARNING: Model exceeds 1M parameter target!")
    
    def forward(self, spec: torch.Tensor) -> torch.Tensor:
        """Forward pass.
        
        Args:
            spec: [B, Freq, T] complex input spectrum
        
        Returns:
            enhanced: [B, Freq, T] complex enhanced spectrum
        """
        # Get magnitude
        mag = torch.abs(spec)
        
        # ERB encoding
        features = self.erb_encoder(mag)  # [B, hidden_dim, T]
        
        # Temporal modeling
        features = self.temporal_gru(features)  # [B, hidden_dim, T]
        
        # Deep filtering
        enhanced = self.deep_filter(features, spec)
        
        return enhanced

## Dataset

In [4]:
def _load_wav(path: str, target_sr: int) -> torch.Tensor:
    """Load audio file and resample if needed."""
    if _HAS_SOUNDFILE:
        data, sr = sf.read(path, dtype="float32", always_2d=True)
        if sr != target_sr:
            if _HAS_TORCHAUDIO:
                wav = torch.from_numpy(np.asarray(data).T)
                wav = torchaudio.functional.resample(wav, sr, target_sr)
            else:
                raise ValueError(f"Sample rate mismatch: {sr} vs {target_sr}")
        else:
            wav = torch.from_numpy(np.asarray(data).T)
    elif _HAS_TORCHAUDIO:
        wav, sr = torchaudio.load(path)
        if sr != target_sr:
            wav = torchaudio.functional.resample(wav, sr, target_sr)
        wav = wav.to(torch.float32)
    else:
        raise ModuleNotFoundError("Install soundfile or torchaudio")
    
    # Mix to mono
    if wav.dim() == 2 and wav.shape[0] > 1:
        wav = wav.mean(dim=0)
    elif wav.dim() == 2:
        wav = wav.squeeze(0)
    
    return wav


class MVDRDataset(Dataset):
    """Dataset for noisy-to-clean pairs (same filenames in both directories)."""
    def __init__(
        self,
        noisy_dir: str,
        clean_dir: str,
        sample_rate: int = 16000,
    ):
        self.noisy_dir = noisy_dir
        self.clean_dir = clean_dir
        self.sr = sample_rate
        
        # Get list of noisy files
        self.files = sorted([f for f in os.listdir(noisy_dir) if f.lower().endswith(".wav")])
        
        # Verify all pairs exist
        missing = []
        for f in self.files:
            clean_path = os.path.join(clean_dir, f)
            if not os.path.exists(clean_path):
                missing.append(f)
        
        if missing:
            raise FileNotFoundError(f"Missing clean files for: {missing[:5]}{'...' if len(missing) > 5 else ''}")
        
        print(f"Loaded {len(self.files)} audio pairs from {noisy_dir}")
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        filename = self.files[idx]
        noisy_path = os.path.join(self.noisy_dir, filename)
        clean_path = os.path.join(self.clean_dir, filename)
        
        noisy_wav = _load_wav(noisy_path, self.sr)
        clean_wav = _load_wav(clean_path, self.sr)
        
        # Match lengths
        min_len = min(noisy_wav.shape[-1], clean_wav.shape[-1])
        noisy_wav = noisy_wav[:min_len]
        clean_wav = clean_wav[:min_len]
        
        # Normalize
        scale = noisy_wav.std() + 1e-8
        noisy_wav = noisy_wav / scale
        clean_wav = clean_wav / scale
        
        return noisy_wav, clean_wav

## SDR Loss Function

Using only SDR (Signal-to-Distortion Ratio) as requested for cleaner optimization.

In [5]:
def sdr_loss(est: torch.Tensor, ref: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Scale-Invariant SDR loss (negative SDR for minimization).
    
    Args:
        est: Estimated signal [B, T]
        ref: Reference signal [B, T]
    
    Returns:
        loss: Negative SI-SDR (lower is better)
    """
    # Zero-mean
    ref = ref - ref.mean(dim=-1, keepdim=True)
    est = est - est.mean(dim=-1, keepdim=True)
    
    # Compute SI-SDR
    dot = torch.sum(est * ref, dim=-1, keepdim=True)
    s_target = (dot / (torch.sum(ref ** 2, dim=-1, keepdim=True) + eps)) * ref
    e_noise = est - s_target
    
    si_sdr = torch.sum(s_target ** 2, dim=-1) / (torch.sum(e_noise ** 2, dim=-1) + eps)
    si_sdr_db = 10.0 * torch.log10(si_sdr + eps)
    
    return -si_sdr_db.mean()

## Training Configuration (V2 Updates)

In [6]:
# ==================== CONFIGURATION V2 ====================

# Dataset paths (pre-split directories)
TRAIN_NOISY_DIR = '/kaggle/input/post-mvdr-snr-5db-no-reverberation-100k/prepared_dataset/train/noisy/'
TRAIN_CLEAN_DIR = '/kaggle/input/post-mvdr-snr-5db-no-reverberation-100k/prepared_dataset/train/clean/'
VAL_NOISY_DIR = '/kaggle/input/post-mvdr-snr-5db-no-reverberation-100k/prepared_dataset/val/noisy/'
VAL_CLEAN_DIR = '/kaggle/input/post-mvdr-snr-5db-no-reverberation-100k/prepared_dataset/val/clean/'
TEST_NOISY_DIR = '/kaggle/input/post-mvdr-snr-5db-no-reverberation-100k/prepared_dataset/test/noisy/'
TEST_CLEAN_DIR = '/kaggle/input/post-mvdr-snr-5db-no-reverberation-100k/prepared_dataset/test/clean/'

SAMPLE_RATE = 16000

# Training hyperparameters (V2 updates)
BATCH_SIZE = 12           # V2: reduced from 16 for longer segments
EPOCHS = 200              # V2: increased from 150
BASE_LR = 3e-4
MIN_LR = 1e-6
WEIGHT_DECAY = 5e-5       # V2: reduced from 1e-4

# Learning rate schedule
WARMUP_EPOCHS = 5
LR_SCHEDULE = 'cosine'

SEED = 42
NUM_WORKERS = 4 if DEVICE == 'cuda' else 0

# Segment length for training (V2 update)
SEGMENT_SECONDS = 3.0     # V2: increased from 2.0
SEGMENT_SAMPLES = int(SAMPLE_RATE * SEGMENT_SECONDS)

# Model architecture (V2 updates)
MODEL_CONFIG = {
    'n_fft': N_FFT,
    'n_erb_bands': 40,    # V2: increased from 32
    'hidden_dim': 80,     # V2: increased from 64
    'gru_dim': 128,       # V2: increased from 96
    'gru_layers': 2,
    'df_order': 5,        # V2: increased from 3
    'df_bins': 128,       # V2: increased from 96
    'dropout': 0.1,
}

# Checkpointing
CHECKPOINT_DIR = 'checkpoints_dfnet_v2'
CHECKPOINT_PREFIX = 'dfnet_v2'
SAVE_EVERY_EPOCHS = 1
AUTO_RESUME = True

# Early stopping
EARLY_STOPPING = True
EARLY_STOP_PATIENCE = 25  # V2: increased from 20
EARLY_STOP_MIN_DELTA_DB = 0.05

# ==========================================================

## Training Utilities

In [7]:
def _seed_everything(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _train_collate(batch):
    """Collate with random cropping for training."""
    noisy_list, clean_list = zip(*batch)
    noisy_out, clean_out = [], []
    
    for noisy, clean in zip(noisy_list, clean_list):
        length = noisy.shape[-1]
        if length >= SEGMENT_SAMPLES:
            start = torch.randint(0, length - SEGMENT_SAMPLES + 1, (1,)).item()
            noisy = noisy[start:start + SEGMENT_SAMPLES]
            clean = clean[start:start + SEGMENT_SAMPLES]
        else:
            pad = SEGMENT_SAMPLES - length
            noisy = F.pad(noisy, (0, pad))
            clean = F.pad(clean, (0, pad))
        
        noisy_out.append(noisy)
        clean_out.append(clean)
    
    return torch.stack(noisy_out), torch.stack(clean_out)


def _val_collate(batch):
    """Collate with center cropping for validation."""
    noisy_list, clean_list = zip(*batch)
    noisy_out, clean_out = [], []
    
    for noisy, clean in zip(noisy_list, clean_list):
        length = noisy.shape[-1]
        if length >= SEGMENT_SAMPLES:
            start = (length - SEGMENT_SAMPLES) // 2
            noisy = noisy[start:start + SEGMENT_SAMPLES]
            clean = clean[start:start + SEGMENT_SAMPLES]
        else:
            pad = SEGMENT_SAMPLES - length
            noisy = F.pad(noisy, (0, pad))
            clean = F.pad(clean, (0, pad))
        
        noisy_out.append(noisy)
        clean_out.append(clean)
    
    return torch.stack(noisy_out), torch.stack(clean_out)


def make_loaders():
    """Create data loaders from pre-split directories."""
    # Create datasets for each split
    train_dataset = MVDRDataset(TRAIN_NOISY_DIR, TRAIN_CLEAN_DIR, SAMPLE_RATE)
    val_dataset = MVDRDataset(VAL_NOISY_DIR, VAL_CLEAN_DIR, SAMPLE_RATE)
    test_dataset = MVDRDataset(TEST_NOISY_DIR, TEST_CLEAN_DIR, SAMPLE_RATE)
    
    print(f'Dataset sizes: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}')
    
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'),
        collate_fn=_train_collate,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'),
        collate_fn=_val_collate,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=1, shuffle=False,
        num_workers=NUM_WORKERS,
    )
    
    return train_loader, val_loader, test_loader

## Learning Rate Scheduler

Warmup + Cosine Annealing to avoid early plateau.

In [8]:
class WarmupCosineScheduler:
    """Learning rate scheduler with warmup and cosine annealing."""
    def __init__(
        self,
        optimizer: optim.Optimizer,
        warmup_epochs: int,
        total_epochs: int,
        base_lr: float,
        min_lr: float,
    ):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.current_epoch = 0
    
    def step(self, epoch: int = None):
        if epoch is not None:
            self.current_epoch = epoch
        else:
            self.current_epoch += 1
        
        if self.current_epoch <= self.warmup_epochs:
            # Linear warmup
            lr = self.base_lr * (self.current_epoch / self.warmup_epochs)
        else:
            # Cosine annealing
            progress = (self.current_epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        
        return lr
    
    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']

## Training Loop

In [9]:
def _make_pbar(iterable, desc: str):
    return tqdm(iterable, desc=desc, leave=True, miniters=1, dynamic_ncols=True)


def forward_pass(model, noisy_wav, clean_wav):
    """Forward pass through model."""
    # STFT
    noisy_spec = stft(noisy_wav, DEVICE)  # [B, F, T]
    
    # Model forward
    enhanced_spec = model(noisy_spec)
    
    # iSTFT
    enhanced_wav = istft(enhanced_spec, DEVICE, length=noisy_wav.shape[-1])
    
    # Compute SDR loss
    loss = sdr_loss(enhanced_wav, clean_wav)
    
    # Compute baseline SDR for comparison
    with torch.no_grad():
        baseline_loss = sdr_loss(noisy_wav, clean_wav)
    
    return loss, baseline_loss, enhanced_wav


def train_one_epoch(model, optimizer, train_loader, epoch_idx):
    model.train()
    
    total_loss = 0.0
    total_baseline = 0.0
    n = 0
    
    pbar = _make_pbar(train_loader, desc=f'Train {epoch_idx:03d}')
    for noisy_wav, clean_wav in pbar:
        noisy_wav = noisy_wav.to(DEVICE)
        clean_wav = clean_wav.to(DEVICE)
        
        loss, baseline_loss, _ = forward_pass(model, noisy_wav, clean_wav)
        
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        
        total_loss += loss.item()
        total_baseline += baseline_loss.item()
        n += 1
        
        enh_sdr = -total_loss / n
        base_sdr = -total_baseline / n
        imp = enh_sdr - base_sdr
        
        pbar.set_postfix(
            sdr=f'{enh_sdr:.2f}dB',
            base=f'{base_sdr:.2f}dB',
            imp=f'{imp:.2f}dB',
            lr=f'{optimizer.param_groups[0]["lr"]:.1e}',
        )
    
    return total_loss / n, total_baseline / n


@torch.no_grad()
def validate(model, val_loader, epoch_idx):
    model.eval()
    
    total_loss = 0.0
    total_baseline = 0.0
    n = 0
    
    pbar = _make_pbar(val_loader, desc=f'Val   {epoch_idx:03d}')
    for noisy_wav, clean_wav in pbar:
        noisy_wav = noisy_wav.to(DEVICE)
        clean_wav = clean_wav.to(DEVICE)
        
        loss, baseline_loss, _ = forward_pass(model, noisy_wav, clean_wav)
        
        total_loss += loss.item()
        total_baseline += baseline_loss.item()
        n += 1
        
        enh_sdr = -total_loss / n
        base_sdr = -total_baseline / n
        imp = enh_sdr - base_sdr
        
        pbar.set_postfix(
            sdr=f'{enh_sdr:.2f}dB',
            base=f'{base_sdr:.2f}dB',
            imp=f'{imp:.2f}dB',
        )
    
    return total_loss / n, total_baseline / n

## Checkpointing

In [10]:
def save_checkpoint(epoch: int, model: nn.Module, optimizer: optim.Optimizer, scheduler, best_sdr: float):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    
    payload = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_epoch': scheduler.current_epoch,
        'best_sdr': best_sdr,
        'model_config': MODEL_CONFIG,
    }
    
    epoch_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_epoch_{epoch:03d}.pt')
    latest_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_latest.pt')
    
    torch.save(payload, epoch_path)
    torch.save(payload, latest_path)
    
    return epoch_path


def find_latest_checkpoint():
    latest_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_latest.pt')
    if os.path.isfile(latest_path):
        return latest_path
    
    pattern = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_epoch_*.pt')
    candidates = glob.glob(pattern)
    if not candidates:
        return None
    
    best = None
    best_epoch = -1
    for p in candidates:
        m = re.search(r'_epoch_(\d+)\.pt$', os.path.basename(p))
        if m:
            ep = int(m.group(1))
            if ep > best_epoch:
                best_epoch = ep
                best = p
    return best


def maybe_resume(model, optimizer, scheduler):
    if not AUTO_RESUME:
        return 1, -float('inf')
    
    ckpt_path = find_latest_checkpoint()
    if ckpt_path is None:
        return 1, -float('inf')
    
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.current_epoch = ckpt.get('scheduler_epoch', 0)
    best_sdr = ckpt.get('best_sdr', -float('inf'))
    last_epoch = ckpt['epoch']
    
    print(f'Resumed from {ckpt_path} (epoch {last_epoch}, best_sdr={best_sdr:.2f}dB)')
    return last_epoch + 1, best_sdr

## Main Training Function

In [11]:
def main():
    _seed_everything(SEED)
    
    # Create data loaders
    train_loader, val_loader, test_loader = make_loaders()
    
    # Create model with V2 config
    model = DeepFilterNetLightV2(**MODEL_CONFIG).to(DEVICE)
    
    # Optimizer with reduced weight decay
    optimizer = optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    
    # Learning rate scheduler
    scheduler = WarmupCosineScheduler(
        optimizer,
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=EPOCHS,
        base_lr=BASE_LR,
        min_lr=MIN_LR,
    )
    
    # Resume if checkpoint exists
    start_epoch, best_val_sdr = maybe_resume(model, optimizer, scheduler)
    
    print(f'\n{"="*60}')
    print(f'DeepFilterNet-Light V2 Training')
    print(f'{"="*60}')
    print(f'Device: {DEVICE}')
    print(f'Train/Val/Test: {len(train_loader.dataset)}/{len(val_loader.dataset)}/{len(test_loader.dataset)}')
    print(f'Segment length: {SEGMENT_SECONDS}s ({SEGMENT_SAMPLES} samples)')
    print(f'Batch size: {BATCH_SIZE}')
    print(f'Start epoch: {start_epoch} / Total: {EPOCHS}')
    print(f'Best val SDR so far: {best_val_sdr:.2f}dB')
    print(f'{"="*60}\n')
    
    if start_epoch > EPOCHS:
        print('Training already complete.')
        return
    
    epochs_no_improve = 0
    
    for epoch in range(start_epoch, EPOCHS + 1):
        # Update learning rate
        lr = scheduler.step(epoch)
        
        # Train
        tr_loss, tr_base = train_one_epoch(model, optimizer, train_loader, epoch)
        
        # Validate
        va_loss, va_base = validate(model, val_loader, epoch)
        
        # Compute metrics
        tr_sdr = -tr_loss
        va_sdr = -va_loss
        tr_base_sdr = -tr_base
        va_base_sdr = -va_base
        tr_imp = tr_sdr - tr_base_sdr
        va_imp = va_sdr - va_base_sdr
        
        print(
            f'Epoch {epoch:03d} | lr={lr:.1e} | '
            f'train: sdr={tr_sdr:.2f}dB, imp={tr_imp:.2f}dB | '
            f'val: sdr={va_sdr:.2f}dB, imp={va_imp:.2f}dB'
        )
        
        # Save checkpoint
        if epoch % SAVE_EVERY_EPOCHS == 0:
            path = save_checkpoint(epoch, model, optimizer, scheduler, best_val_sdr)
            print(f'Saved: {path}')
        
        # Early stopping
        if EARLY_STOPPING:
            if va_sdr > best_val_sdr + EARLY_STOP_MIN_DELTA_DB:
                best_val_sdr = va_sdr
                epochs_no_improve = 0
                # Save best model
                best_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_best.pt')
                torch.save({
                    'model_state': model.state_dict(),
                    'model_config': MODEL_CONFIG,
                    'best_sdr': best_val_sdr,
                    'epoch': epoch,
                }, best_path)
                print(f'New best model: {va_sdr:.2f}dB')
            else:
                epochs_no_improve += 1
            
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f'Early stopping at epoch {epoch}. Best val SDR: {best_val_sdr:.2f}dB')
                break
    
    print(f'\nTraining complete. Best val SDR: {best_val_sdr:.2f}dB')
    return test_loader

## Inference

In [12]:
def _save_wav(path: str, wav: torch.Tensor, sr: int):
    """Save wav tensor to file."""
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    wav = wav.detach().cpu().to(torch.float32).view(-1)
    
    if _HAS_SOUNDFILE:
        sf.write(path, wav.numpy(), sr)
    elif _HAS_TORCHAUDIO:
        torchaudio.save(path, wav.unsqueeze(0), sr)
    else:
        raise ModuleNotFoundError("Install soundfile or torchaudio")


def load_model_from_checkpoint(checkpoint_path: str, device: str = DEVICE) -> nn.Module:
    """Load model from checkpoint."""
    ckpt = torch.load(checkpoint_path, map_location=device)
    
    # Get model config from checkpoint or use default
    config = ckpt.get('model_config', MODEL_CONFIG)
    
    model = DeepFilterNetLightV2(**config).to(device)
    
    state = ckpt['model_state'] if 'model_state' in ckpt else ckpt
    model.load_state_dict(state)
    model.eval()
    return model


@torch.no_grad()
def enhance_waveform(wav: torch.Tensor, model: nn.Module, device: str = DEVICE) -> torch.Tensor:
    """Enhance a mono waveform."""
    if wav.dim() != 1:
        wav = wav.view(-1)
    
    wav = wav.to(device)
    scale = wav.std() + 1e-8
    wav_norm = wav / scale
    
    # STFT
    spec = stft(wav_norm.unsqueeze(0), device)
    
    # Enhance
    enhanced_spec = model(spec)
    
    # iSTFT
    enhanced = istft(enhanced_spec, device, length=wav_norm.shape[-1]).squeeze(0)
    
    return enhanced * scale


def enhance_directory(input_dir: str, checkpoint_path: str, output_dir: str, pattern: str = '*.wav'):
    """Enhance all wav files in a directory."""
    model = load_model_from_checkpoint(checkpoint_path)
    paths = sorted(glob.glob(os.path.join(input_dir, pattern)))
    
    if not paths:
        raise FileNotFoundError(f'No files matching {pattern} in {input_dir}')
    
    os.makedirs(output_dir, exist_ok=True)
    
    for p in tqdm(paths, desc='Enhancing'):
        wav = _load_wav(p, SAMPLE_RATE)
        enhanced = enhance_waveform(wav, model)
        
        base = os.path.splitext(os.path.basename(p))[0]
        out_path = os.path.join(output_dir, f'{base}_enhanced.wav')
        _save_wav(out_path, enhanced, SAMPLE_RATE)
    
    print(f'Saved enhanced files to: {output_dir}')

In [13]:
# # Example usage:
checkpoint_path = '/Users/emonchowdhury/Desktop/Phase 2/av_zoom/audio/models/custom_model_2/checkpoints_dfnet/v2_new_reverb/dfnet_v2_best.pt'
enhance_directory('/Users/emonchowdhury/Desktop/final sample/Reverb(Uncompensated)/MVDR+PF', checkpoint_path, '/Users/emonchowdhury/Desktop/final sample/enhanced')

DeepFilterNetLightV2: 0.383M parameters


FileNotFoundError: No files matching *.wav in /Users/emonchowdhury/Desktop/final sample/Reverb(Uncompensated)/MVDR+PF

In [14]:
@torch.no_grad()
def evaluate_test_set(checkpoint_path: str, noisy_dir: str = TEST_NOISY_DIR, clean_dir: str = TEST_CLEAN_DIR):
    """Evaluate model on test set."""
    # Create test dataset
    test_dataset = MVDRDataset(noisy_dir, clean_dir, SAMPLE_RATE)
    print(f'Evaluating on {len(test_dataset)} test samples')
    
    model = load_model_from_checkpoint(checkpoint_path)
    
    results = []
    total_base_sdr = 0.0
    total_enh_sdr = 0.0
    
    for idx in tqdm(range(len(test_dataset)), desc='Testing'):
        noisy_wav, clean_wav = test_dataset[idx]
        noisy_wav = noisy_wav.to(DEVICE)
        clean_wav = clean_wav.to(DEVICE)
        
        # Baseline SDR
        base_loss = sdr_loss(noisy_wav.unsqueeze(0), clean_wav.unsqueeze(0))
        base_sdr = -base_loss.item()
        
        # Enhanced SDR
        enhanced = enhance_waveform(noisy_wav, model)
        enh_loss = sdr_loss(enhanced.unsqueeze(0), clean_wav.unsqueeze(0))
        enh_sdr = -enh_loss.item()
        
        results.append({
            'file': test_dataset.files[idx],
            'base_sdr': base_sdr,
            'enh_sdr': enh_sdr,
            'improvement': enh_sdr - base_sdr,
        })
        
        total_base_sdr += base_sdr
        total_enh_sdr += enh_sdr
    
    n = len(results)
    avg_base = total_base_sdr / n
    avg_enh = total_enh_sdr / n
    avg_imp = avg_enh - avg_base
    
    print(f'\n{"="*50}')
    print(f'TEST SET RESULTS V2 ({n} samples)')
    print(f'{"="*50}')
    print(f'  Baseline SDR:  {avg_base:.2f} dB')
    print(f'  Enhanced SDR:  {avg_enh:.2f} dB')
    print(f'  Improvement:   {avg_imp:.2f} dB')
    print(f'{"="*50}')
    
    return {
        'n_samples': n,
        'avg_baseline_sdr': avg_base,
        'avg_enhanced_sdr': avg_enh,
        'avg_improvement': avg_imp,
        'per_sample': results,
    }

## Run Training

In [15]:
# test_loader = main()

## Latency Benchmark — Real-Time Smartphone Feasibility

Measures end-to-end inference latency (STFT → model → iSTFT) on **CPU** to simulate smartphone conditions.

**Key metrics:**
- **Per-frame latency**: Time to process one STFT hop (8 ms at 16 kHz / 128 hop)
- **Real-Time Factor (RTF)**: `processing_time / audio_duration` — must be **< 1.0** for real-time
- **Estimated smartphone multiplier**: ~2–4× slower than desktop CPU

In [16]:
import time


def benchmark_latency(
    checkpoint_path: str,
    durations_sec: List[float] = [0.5, 1.0, 2.0, 3.0, 5.0],
    n_warmup: int = 5,
    n_runs: int = 50,
    smartphone_slowdown: float = 3.0,
):
    """Benchmark model inference latency on CPU to estimate smartphone feasibility.
    
    Args:
        checkpoint_path: Path to model checkpoint.
        durations_sec: Audio durations to test (in seconds).
        n_warmup: Number of warmup iterations (excluded from timing).
        n_runs: Number of timed iterations per duration.
        smartphone_slowdown: Estimated CPU slowdown factor vs desktop
                             (typical smartphone is 2-4x slower).
    
    Returns:
        Dictionary with latency statistics for each duration.
    """
    # Always benchmark on CPU to simulate smartphone
    device = 'cpu'
    model = load_model_from_checkpoint(checkpoint_path, device=device)
    model.eval()
    
    hop_duration_ms = (HOP_LENGTH / SAMPLE_RATE) * 1000  # ms per STFT frame
    
    print(f'{"=" * 70}')
    print(f'  DeepFilterNet-Light V2 — Latency Benchmark (CPU)')
    print(f'{"=" * 70}')
    print(f'  STFT config: n_fft={N_FFT}, hop={HOP_LENGTH}, sr={SAMPLE_RATE}')
    print(f'  Frame duration (real-time budget per hop): {hop_duration_ms:.1f} ms')
    print(f'  Warmup iters: {n_warmup} | Timed iters: {n_runs}')
    print(f'  Smartphone slowdown factor: {smartphone_slowdown:.1f}x')
    print(f'{"=" * 70}\n')
    
    all_results = {}
    
    for dur in durations_sec:
        n_samples = int(SAMPLE_RATE * dur)
        n_frames = (n_samples + HOP_LENGTH - 1) // HOP_LENGTH
        
        # Synthetic noisy input
        wav = torch.randn(1, n_samples, device=device)
        
        # ---------- Warmup ----------
        for _ in range(n_warmup):
            spec = stft(wav, device)
            enh = model(spec)
            _ = istft(enh, device, length=n_samples)
        
        # ---------- Full-pipeline timing ----------
        full_times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            spec = stft(wav, device)
            enh = model(spec)
            _ = istft(enh, device, length=n_samples)
            t1 = time.perf_counter()
            full_times.append((t1 - t0) * 1000)  # ms
        
        # ---------- Model-only timing (excl. STFT/iSTFT) ----------
        spec = stft(wav, device)
        model_times = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            _ = model(spec)
            t1 = time.perf_counter()
            model_times.append((t1 - t0) * 1000)
        
        full_arr = np.array(full_times)
        model_arr = np.array(model_times)
        
        audio_dur_ms = dur * 1000
        rtf = np.mean(full_arr) / audio_dur_ms
        rtf_smartphone = rtf * smartphone_slowdown
        per_frame_ms = np.mean(full_arr) / n_frames
        per_frame_smartphone = per_frame_ms * smartphone_slowdown
        
        result = {
            'audio_duration_s': dur,
            'n_samples': n_samples,
            'n_frames': n_frames,
            # Full pipeline (STFT + model + iSTFT)
            'full_mean_ms': np.mean(full_arr),
            'full_std_ms': np.std(full_arr),
            'full_p50_ms': np.percentile(full_arr, 50),
            'full_p95_ms': np.percentile(full_arr, 95),
            'full_p99_ms': np.percentile(full_arr, 99),
            # Model only
            'model_mean_ms': np.mean(model_arr),
            'model_std_ms': np.std(model_arr),
            'model_p95_ms': np.percentile(model_arr, 95),
            # Per-frame
            'per_frame_ms': per_frame_ms,
            'per_frame_smartphone_ms': per_frame_smartphone,
            'frame_budget_ms': hop_duration_ms,
            # Real-Time Factor
            'rtf_cpu': rtf,
            'rtf_smartphone_est': rtf_smartphone,
        }
        all_results[dur] = result
        
        realtime_ok = rtf_smartphone < 1.0
        frame_ok = per_frame_smartphone < hop_duration_ms
        
        status = '✅ REAL-TIME OK' if realtime_ok else '❌ TOO SLOW'
        frame_status = '✅' if frame_ok else '❌'
        
        print(f'--- {dur:.1f}s audio ({n_samples} samples, {n_frames} frames) ---')
        print(f'  Full pipeline : {np.mean(full_arr):7.2f} ± {np.std(full_arr):.2f} ms  '
              f'(p95={np.percentile(full_arr, 95):.2f}, p99={np.percentile(full_arr, 99):.2f})')
        print(f'  Model only    : {np.mean(model_arr):7.2f} ± {np.std(model_arr):.2f} ms  '
              f'(p95={np.percentile(model_arr, 95):.2f})')
        print(f'  Per-frame     : {per_frame_ms:7.2f} ms (desktop)  |  '
              f'{per_frame_smartphone:.2f} ms (smartphone est.)  '
              f'[budget: {hop_duration_ms:.1f} ms] {frame_status}')
        print(f'  RTF (desktop) : {rtf:.4f}  |  '
              f'RTF (smartphone): {rtf_smartphone:.4f}  {status}')
        print()
    
    # ---------- Summary ----------
    print(f'{"=" * 70}')
    print(f'  SUMMARY')
    print(f'{"=" * 70}')
    print(f'  {"Duration":>8s}  {"Full (ms)":>10s}  {"Model (ms)":>10s}  '
          f'{"RTF-CPU":>8s}  {"RTF-Phone":>10s}  {"Status"}')
    print(f'  {"-"*8}  {"-"*10}  {"-"*10}  {"-"*8}  {"-"*10}  {"-"*10}')
    
    for dur, r in all_results.items():
        ok = '✅ OK' if r['rtf_smartphone_est'] < 1.0 else '❌ SLOW'
        print(f'  {dur:>7.1f}s  {r["full_mean_ms"]:>10.2f}  {r["model_mean_ms"]:>10.2f}  '
              f'{r["rtf_cpu"]:>8.4f}  {r["rtf_smartphone_est"]:>10.4f}  {ok}')
    
    avg_rtf_phone = np.mean([r['rtf_smartphone_est'] for r in all_results.values()])
    print(f'\n  Average smartphone RTF: {avg_rtf_phone:.4f}')
    if avg_rtf_phone < 1.0:
        headroom = (1.0 - avg_rtf_phone) * 100
        print(f'  ✅ Model is suitable for real-time smartphone deployment ({headroom:.0f}% headroom)')
    else:
        overshoot = (avg_rtf_phone - 1.0) * 100
        print(f'  ❌ Model is {overshoot:.0f}% too slow for real-time smartphone deployment')
        print(f'     Consider reducing model size or using quantization / NNAPI delegation.')
    print(f'{"=" * 70}')
    
    return all_results

In [17]:
# Run latency benchmark
latency_results = benchmark_latency(
    checkpoint_path='checkpoints_dfnet/v2/dfnet_v2_best.pt',
    durations_sec=[0.5, 1.0, 2.0, 3.0, 5.0],
    n_warmup=5,
    n_runs=50,
    smartphone_slowdown=3.0,  # Typical ARM vs desktop CPU factor
)

DeepFilterNetLightV2: 0.383M parameters
  DeepFilterNet-Light V2 — Latency Benchmark (CPU)
  STFT config: n_fft=512, hop=128, sr=16000
  Frame duration (real-time budget per hop): 8.0 ms
  Warmup iters: 5 | Timed iters: 50
  Smartphone slowdown factor: 3.0x

--- 0.5s audio (8000 samples, 63 frames) ---
  Full pipeline :    5.80 ± 0.56 ms  (p95=6.53, p99=7.72)
  Model only    :    4.65 ± 0.25 ms  (p95=5.16)
  Per-frame     :    0.09 ms (desktop)  |  0.28 ms (smartphone est.)  [budget: 8.0 ms] ✅
  RTF (desktop) : 0.0116  |  RTF (smartphone): 0.0348  ✅ REAL-TIME OK

--- 1.0s audio (16000 samples, 125 frames) ---
  Full pipeline :   11.53 ± 0.52 ms  (p95=12.27, p99=12.41)
  Model only    :    9.09 ± 0.46 ms  (p95=9.73)
  Per-frame     :    0.09 ms (desktop)  |  0.28 ms (smartphone est.)  [budget: 8.0 ms] ✅
  RTF (desktop) : 0.0115  |  RTF (smartphone): 0.0346  ✅ REAL-TIME OK

--- 2.0s audio (32000 samples, 250 frames) ---
  Full pipeline :   21.65 ± 6.50 ms  (p95=22.21, p99=45.24)
  Model 

## Combined Pipeline Report: MVDR + DeepFilterNet + Video

### Workflow
1. **MATLAB**: Set `K_selected` in `benchmark_mvdr_latency.m` → run → copy the two output lines
2. **Below**: Paste `mvdr_K` and `mvdr_ms_per_frame` → run → get final combined report

### Pipeline (MVDR N=512 aligned — no redundant STFT/iSTFT)
```
Mic Array → STFT(N=512) → MVDR beamform → DeepFilterNet model → iSTFT → Speaker
                               ↑ shared STFT — no redundant transforms ↑
```

In [18]:
def combined_pipeline_report(
    checkpoint_path: str,
    # --- PASTE FROM MATLAB ---
    mvdr_K: int = 4,
    mvdr_ms_per_frame: float = 0.8511,
    # --- Estimation factors ---
    mvdr_smartphone_factor: float = 3.0,
    model_smartphone_factor: float = 3.0,
    # --- Video pipeline ---
    video_fps: int = 30,
    video_frame_processing_ms: float = 10.0,
    # --- Benchmark config ---
    n_warmup: int = 5,
    n_runs: int = 50,
    # --- Threading ---
    parallel_av: bool = True,
):
    """Final combined pipeline report.
    
    Takes MVDR per-frame timing (from MATLAB benchmark) + measures DeepFilterNet
    latency here, then produces the full real-time feasibility verdict.
    """
    device = 'cpu'
    model = load_model_from_checkpoint(checkpoint_path, device=device)
    model.eval()
    
    hop_budget_ms = (HOP_LENGTH / SAMPLE_RATE) * 1000  # 8.0 ms
    video_budget_ms = 1000.0 / video_fps               # 33.3 ms @ 30fps
    hops_per_video_frame = video_budget_ms / hop_budget_ms
    
    # =========================================================================
    # 1) Measure DeepFilterNet (model + iSTFT, direct STFT feed from MVDR)
    # =========================================================================
    test_dur = 1.0
    n_samples = int(SAMPLE_RATE * test_dur)
    n_frames = (n_samples + HOP_LENGTH - 1) // HOP_LENGTH
    
    # Simulate MVDR STFT output: complex [1, 257, T]
    spec_mvdr = torch.randn(1, N_FFT // 2 + 1, n_frames, dtype=torch.cfloat, device=device)
    
    for _ in range(n_warmup):
        _ = model(spec_mvdr)
    
    model_times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        enh = model(spec_mvdr)
        _ = istft(enh, device, length=n_samples)
        t1 = time.perf_counter()
        model_times.append((t1 - t0) * 1000)
    
    model_total_ms = np.mean(model_times)
    model_per_frame_desk = model_total_ms / n_frames
    model_per_frame_phone = model_per_frame_desk * model_smartphone_factor
    
    # =========================================================================
    # 2) MVDR phone estimate
    # =========================================================================
    mvdr_phone = mvdr_ms_per_frame * mvdr_smartphone_factor
    
    # =========================================================================
    # 3) Combined
    # =========================================================================
    combined_desk = mvdr_ms_per_frame + model_per_frame_desk
    combined_phone = mvdr_phone + model_per_frame_phone
    audio_ok = combined_phone < hop_budget_ms
    
    if audio_ok:
        headroom = ((hop_budget_ms - combined_phone) / hop_budget_ms) * 100
    else:
        headroom = -((combined_phone - hop_budget_ms) / hop_budget_ms) * 100
    
    # AV feasibility
    audio_per_vframe = combined_phone * hops_per_video_frame
    if parallel_av:
        av_bottleneck = max(audio_per_vframe, video_frame_processing_ms)
        av_ok = av_bottleneck < video_budget_ms
    else:
        av_total = audio_per_vframe + video_frame_processing_ms
        av_ok = av_total < video_budget_ms
    
    # =========================================================================
    # REPORT
    # =========================================================================
    print(f'{"=" * 72}')
    print(f'  COMBINED PIPELINE REPORT: MVDR + DeepFilterNet + Video')
    print(f'  Audiovisual Zooming — Smartphone Real-Time Feasibility')
    print(f'{"=" * 72}')
    
    print(f'\n  Configuration:')
    print(f'    MVDR  : N=512, hop=128, nfft=512, 2 mics, K={mvdr_K}')
    print(f'    Model : DeepFilterNetLightV2, 0.383M params, CPU')
    print(f'    Video : {video_fps} fps, ~{video_frame_processing_ms:.0f} ms/frame')
    print(f'    STFT  : Aligned N=512 (direct feed, no redundant STFT/iSTFT)')
    print(f'    Thread: {"Parallel (audio+video on separate cores)" if parallel_av else "Sequential"}')
    
    print(f'\n  MVDR (from MATLAB benchmark, K={mvdr_K}):')
    print(f'    Desktop : {mvdr_ms_per_frame:.4f} ms/frame')
    print(f'    Phone   : {mvdr_phone:.4f} ms/frame  (×{mvdr_smartphone_factor})')
    
    print(f'\n  DeepFilterNet (measured, {n_frames} frames):')
    print(f'    Desktop : {model_per_frame_desk:.4f} ms/frame')
    print(f'    Phone   : {model_per_frame_phone:.4f} ms/frame  (×{model_smartphone_factor})')
    
    print(f'\n{"─" * 72}')
    print(f'  AUDIO BUDGET (per hop = {hop_budget_ms:.1f} ms)')
    print(f'{"─" * 72}')
    print(f'  {"Component":<32s}  {"Desktop":>10s}  {"Phone":>10s}  {"Budget":>8s}')
    print(f'  {"─"*32}  {"─"*10}  {"─"*10}  {"─"*8}')
    print(f'  {"MVDR (K=" + str(mvdr_K) + ")":<32s}  {mvdr_ms_per_frame:>9.4f}  {mvdr_phone:>9.4f}  {hop_budget_ms:>7.1f}')
    print(f'  {"DeepFilterNet (model+iSTFT)":<32s}  {model_per_frame_desk:>9.4f}  {model_per_frame_phone:>9.4f}  {hop_budget_ms:>7.1f}')
    print(f'  {"─"*32}  {"─"*10}  {"─"*10}  {"─"*8}')
    print(f'  {"COMBINED AUDIO":<32s}  {combined_desk:>9.4f}  {combined_phone:>9.4f}  {hop_budget_ms:>7.1f}')
    print(f'  Headroom: {headroom:+.0f}%  {"✅ REAL-TIME" if audio_ok else "❌ OVER BUDGET"}')
    
    # AV section
    print(f'\n{"─" * 72}')
    print(f'  AV BUDGET (per video frame = {video_budget_ms:.1f} ms @ {video_fps}fps)')
    print(f'{"─" * 72}')
    print(f'  Audio ({hops_per_video_frame:.1f} hops/vframe) : {audio_per_vframe:.2f} ms')
    print(f'  Video processing            : {video_frame_processing_ms:.2f} ms')
    if parallel_av:
        print(f'  Parallel max (bottleneck)   : {av_bottleneck:.2f} ms / {video_budget_ms:.1f} ms  '
              f'{"✅" if av_ok else "❌"}')
    else:
        print(f'  Sequential total            : {av_total:.2f} ms / {video_budget_ms:.1f} ms  '
              f'{"✅" if av_ok else "❌"}')
    
    # Latency split bar
    mvdr_pct = mvdr_phone / combined_phone * 100
    model_pct = 100 - mvdr_pct
    bar_w = 50
    mvdr_bar = max(1, int(mvdr_pct / 100 * bar_w))
    model_bar = bar_w - mvdr_bar
    used_pct = combined_phone / hop_budget_ms * 100
    used_bar = min(bar_w, max(1, int(used_pct / 100 * bar_w)))
    free_bar = bar_w - used_bar
    
    print(f'\n  ⚡ Latency Split (phone):')
    print(f'    MVDR  [{"█" * mvdr_bar}{"░" * model_bar}] {mvdr_pct:.0f}% ({mvdr_phone:.4f} ms)')
    print(f'    Model [{"░" * mvdr_bar}{"█" * model_bar}] {model_pct:.0f}% ({model_per_frame_phone:.4f} ms)')
    print(f'    Total [{"█" * used_bar}{"░" * free_bar}] {combined_phone:.4f} / {hop_budget_ms:.1f} ms ({headroom:+.0f}%)')
    
    # Final verdict
    print(f'\n{"=" * 72}')
    print(f'  VERDICT')
    print(f'{"=" * 72}')
    
    if audio_ok and av_ok:
        print(f'\n  ✅ REAL-TIME FEASIBLE')
        print(f'     Combined audio: {combined_phone:.4f} ms/frame ({headroom:+.0f}% headroom)')
        print(f'     Audio RTF (phone): {combined_phone / hop_budget_ms:.4f}')
        print(f'     MVDR weights update every {mvdr_K} frames = every {mvdr_K * hop_budget_ms:.0f} ms')
        if mvdr_K <= 1:
            print(f'\n  🎯 K=1: Max adaptivity, tracks fast-moving sources in real time')
        elif mvdr_K <= 4:
            print(f'\n  🎯 K={mvdr_K}: Great balance — fast tracking + plenty of headroom')
        elif mvdr_K <= 8:
            print(f'\n  🎯 K={mvdr_K}: Good for slowly-moving or stationary sources')
        else:
            print(f'\n  🎯 K={mvdr_K}: Minimal cost but slower source tracking')
    elif audio_ok and not av_ok:
        print(f'\n  ⚠️  Audio OK but AV pipeline exceeds budget')
        print(f'     Consider: lighter video model or lower fps')
    else:
        print(f'\n  ❌ NOT FEASIBLE with K={mvdr_K}')
        print(f'     Try: larger K, INT8 quantization, CoreML/NNAPI, larger hop')
    
    print(f'\n{"=" * 72}')
    
    return {
        'mvdr_K': mvdr_K,
        'mvdr_desk_ms': mvdr_ms_per_frame,
        'mvdr_phone_ms': mvdr_phone,
        'model_desk_ms': model_per_frame_desk,
        'model_phone_ms': model_per_frame_phone,
        'combined_desk_ms': combined_desk,
        'combined_phone_ms': combined_phone,
        'headroom_pct': headroom,
        'audio_ok': audio_ok,
        'av_ok': av_ok,
    }

In [19]:
# ═══════════════════════════════════════════════════════════
# PASTE FROM MATLAB benchmark_mvdr_latency.m output below:
# ═══════════════════════════════════════════════════════════
mvdr_K = 4
mvdr_ms_per_frame = 0.8511
# ═══════════════════════════════════════════════════════════

pipeline_results = combined_pipeline_report(
    checkpoint_path='checkpoints_dfnet/v2/dfnet_v2_best.pt',
    mvdr_K=mvdr_K,
    mvdr_ms_per_frame=mvdr_ms_per_frame,
)

DeepFilterNetLightV2: 0.383M parameters
  COMBINED PIPELINE REPORT: MVDR + DeepFilterNet + Video
  Audiovisual Zooming — Smartphone Real-Time Feasibility

  Configuration:
    MVDR  : N=512, hop=128, nfft=512, 2 mics, K=4
    Model : DeepFilterNetLightV2, 0.383M params, CPU
    Video : 30 fps, ~10 ms/frame
    STFT  : Aligned N=512 (direct feed, no redundant STFT/iSTFT)
    Thread: Parallel (audio+video on separate cores)

  MVDR (from MATLAB benchmark, K=4):
    Desktop : 0.8511 ms/frame
    Phone   : 2.5533 ms/frame  (×3.0)

  DeepFilterNet (measured, 125 frames):
    Desktop : 0.0858 ms/frame
    Phone   : 0.2573 ms/frame  (×3.0)

────────────────────────────────────────────────────────────────────────
  AUDIO BUDGET (per hop = 8.0 ms)
────────────────────────────────────────────────────────────────────────
  Component                            Desktop       Phone    Budget
  ────────────────────────────────  ──────────  ──────────  ────────
  MVDR (K=4)                           0

## Combined Pipeline Report: MVDR + DTLN + Video

### Workflow
1. **MATLAB**: Set `K_selected` in `benchmark_mvdr_latency.m` → run → copy the two output lines
2. **Below**: Paste `mvdr_K` and `mvdr_ms_per_frame` → run → get final combined report

### Pipeline (per-hop timing, 8 ms budget)
```
Mic Array → STFT(N=512) → MVDR beamform → DTLN (streaming, block=512/hop=128) → Speaker
```

In [23]:
def combined_pipeline_report_dtln(
    saved_model_path: str,
    # --- PASTE FROM MATLAB ---
    mvdr_K: int = 4,
    mvdr_ms_per_frame: float = 0.8511,
    # --- Estimation factors ---
    mvdr_smartphone_factor: float = 3.0,
    model_smartphone_factor: float = 3.0,
    # --- Video pipeline ---
    video_fps: int = 30,
    video_frame_processing_ms: float = 10.0,
    # --- Benchmark config ---
    n_warmup: int = 5,
    n_runs: int = 50,
    # --- Threading ---
    parallel_av: bool = True,
):
    """Final combined pipeline report for MVDR + DTLN + Video."""
    try:
        import tensorflow as tf
    except ModuleNotFoundError as e:
        raise ModuleNotFoundError("TensorFlow is required for DTLN combined report.") from e

    block_len = 512
    block_shift = 128
    sample_rate = 16000

    infer_model = tf.saved_model.load(saved_model_path)
    infer = infer_model.signatures["serving_default"]
    output_key = next(iter(infer.structured_outputs.keys()))

    hop_budget_ms = (block_shift / sample_rate) * 1000  # 8.0 ms
    video_budget_ms = 1000.0 / video_fps               # 33.3 ms @ 30fps
    hops_per_video_frame = video_budget_ms / hop_budget_ms

    # =========================================================================
    # 1) Measure DTLN streaming model per-hop latency
    # =========================================================================
    in_block = np.random.randn(1, block_len).astype(np.float32)

    for _ in range(n_warmup):
        _ = infer(tf.constant(in_block))[output_key]

    model_times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = infer(tf.constant(in_block))[output_key]
        t1 = time.perf_counter()
        model_times.append((t1 - t0) * 1000)

    model_per_frame_desk = float(np.mean(model_times))
    model_per_frame_phone = model_per_frame_desk * model_smartphone_factor

    # =========================================================================
    # 2) MVDR phone estimate
    # =========================================================================
    mvdr_phone = mvdr_ms_per_frame * mvdr_smartphone_factor

    # =========================================================================
    # 3) Combined
    # =========================================================================
    combined_desk = mvdr_ms_per_frame + model_per_frame_desk
    combined_phone = mvdr_phone + model_per_frame_phone
    audio_ok = combined_phone < hop_budget_ms

    if audio_ok:
        headroom = ((hop_budget_ms - combined_phone) / hop_budget_ms) * 100
    else:
        headroom = -((combined_phone - hop_budget_ms) / hop_budget_ms) * 100

    # AV feasibility
    audio_per_vframe = combined_phone * hops_per_video_frame
    if parallel_av:
        av_bottleneck = max(audio_per_vframe, video_frame_processing_ms)
        av_ok = av_bottleneck < video_budget_ms
    else:
        av_total = audio_per_vframe + video_frame_processing_ms
        av_ok = av_total < video_budget_ms

    # =========================================================================
    # REPORT
    # =========================================================================
    print(f'{"=" * 72}')
    print(f'  COMBINED PIPELINE REPORT: MVDR + DTLN + Video')
    print(f'  Audiovisual Zooming — Smartphone Real-Time Feasibility')
    print(f'{"=" * 72}')

    print(f'\n  Configuration:')
    print(f'    MVDR  : N=512, hop=128, nfft=512, 2 mics, K={mvdr_K}')
    print(f'    Model : DTLN SavedModel (streaming), CPU')
    print(f'    Video : {video_fps} fps, ~{video_frame_processing_ms:.0f} ms/frame')
    print(f'    Timing: Per-hop model timing with block=512, hop=128')
    print(f'    Thread: {"Parallel (audio+video on separate cores)" if parallel_av else "Sequential"}')

    print(f'\n  MVDR (from MATLAB benchmark, K={mvdr_K}):')
    print(f'    Desktop : {mvdr_ms_per_frame:.4f} ms/frame')
    print(f'    Phone   : {mvdr_phone:.4f} ms/frame  (×{mvdr_smartphone_factor})')

    print(f'\n  DTLN (measured, per hop):')
    print(f'    Desktop : {model_per_frame_desk:.4f} ms/frame')
    print(f'    Phone   : {model_per_frame_phone:.4f} ms/frame  (×{model_smartphone_factor})')

    print(f'\n{"─" * 72}')
    print(f'  AUDIO BUDGET (per hop = {hop_budget_ms:.1f} ms)')
    print(f'{"─" * 72}')
    print(f'  {"Component":<32s}  {"Desktop":>10s}  {"Phone":>10s}  {"Budget":>8s}')
    print(f'  {"─"*32}  {"─"*10}  {"─"*10}  {"─"*8}')
    print(f'  {"MVDR (K=" + str(mvdr_K) + ")":<32s}  {mvdr_ms_per_frame:>9.4f}  {mvdr_phone:>9.4f}  {hop_budget_ms:>7.1f}')
    print(f'  {"DTLN (streaming block)":<32s}  {model_per_frame_desk:>9.4f}  {model_per_frame_phone:>9.4f}  {hop_budget_ms:>7.1f}')
    print(f'  {"─"*32}  {"─"*10}  {"─"*10}  {"─"*8}')
    print(f'  {"COMBINED AUDIO":<32s}  {combined_desk:>9.4f}  {combined_phone:>9.4f}  {hop_budget_ms:>7.1f}')
    print(f'  Headroom: {headroom:+.0f}%  {"✅ REAL-TIME" if audio_ok else "❌ OVER BUDGET"}')

    print(f'\n{"─" * 72}')
    print(f'  AV BUDGET (per video frame = {video_budget_ms:.1f} ms @ {video_fps}fps)')
    print(f'{"─" * 72}')
    print(f'  Audio ({hops_per_video_frame:.1f} hops/vframe) : {audio_per_vframe:.2f} ms')
    print(f'  Video processing            : {video_frame_processing_ms:.2f} ms')
    if parallel_av:
        print(f'  Parallel max (bottleneck)   : {av_bottleneck:.2f} ms / {video_budget_ms:.1f} ms  '
              f'{"✅" if av_ok else "❌"}')
    else:
        print(f'  Sequential total            : {av_total:.2f} ms / {video_budget_ms:.1f} ms  '
              f'{"✅" if av_ok else "❌"}')

    mvdr_pct = mvdr_phone / combined_phone * 100
    model_pct = 100 - mvdr_pct
    bar_w = 50
    mvdr_bar = max(1, int(mvdr_pct / 100 * bar_w))
    model_bar = bar_w - mvdr_bar
    used_pct = combined_phone / hop_budget_ms * 100
    used_bar = min(bar_w, max(1, int(used_pct / 100 * bar_w)))
    free_bar = bar_w - used_bar

    print(f'\n  ⚡ Latency Split (phone):')
    print(f'    MVDR  [{"█" * mvdr_bar}{"░" * model_bar}] {mvdr_pct:.0f}% ({mvdr_phone:.4f} ms)')
    print(f'    Model [{"░" * mvdr_bar}{"█" * model_bar}] {model_pct:.0f}% ({model_per_frame_phone:.4f} ms)')
    print(f'    Total [{"█" * used_bar}{"░" * free_bar}] {combined_phone:.4f} / {hop_budget_ms:.1f} ms ({headroom:+.0f}%)')

    print(f'\n{"=" * 72}')
    print(f'  VERDICT')
    print(f'{"=" * 72}')

    if audio_ok and av_ok:
        print(f'\n  ✅ REAL-TIME FEASIBLE')
        print(f'     Combined audio: {combined_phone:.4f} ms/frame ({headroom:+.0f}% headroom)')
        print(f'     Audio RTF (phone): {combined_phone / hop_budget_ms:.4f}')
        print(f'     MVDR weights update every {mvdr_K} frames = every {mvdr_K * hop_budget_ms:.0f} ms')
    elif audio_ok and not av_ok:
        print(f'\n  ⚠️  Audio OK but AV pipeline exceeds budget')
        print(f'     Consider: lighter video model or lower fps')
    else:
        print(f'\n  ❌ NOT FEASIBLE with K={mvdr_K}')
        print(f'     Try: larger K, quantization, or hardware acceleration')

    print(f'\n{"=" * 72}')

    return {
        'mvdr_K': mvdr_K,
        'mvdr_desk_ms': mvdr_ms_per_frame,
        'mvdr_phone_ms': mvdr_phone,
        'model_desk_ms': model_per_frame_desk,
        'model_phone_ms': model_per_frame_phone,
        'combined_desk_ms': combined_desk,
        'combined_phone_ms': combined_phone,
        'headroom_pct': headroom,
        'audio_ok': audio_ok,
        'av_ok': av_ok,
    }

In [24]:
# ═══════════════════════════════════════════════════════════
# PASTE FROM MATLAB benchmark_mvdr_latency.m output below:
# ═══════════════════════════════════════════════════════════
mvdr_K = 4
mvdr_ms_per_frame = 0.8511
# ═══════════════════════════════════════════════════════════

pipeline_results_dtln = combined_pipeline_report_dtln(
    saved_model_path='/Users/emonchowdhury/Desktop/Phase 2/av_zoom/audio/models/DTLN-master/pretrained_model/dtln_saved_model',
    mvdr_K=mvdr_K,
    mvdr_ms_per_frame=mvdr_ms_per_frame,
)

ModuleNotFoundError: TensorFlow is required for DTLN combined report.